In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'e:\\ml projects\\fraud-transaction-detection'

In [4]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [5]:
from src.transaction_fraud_detection.utils.common import read_yaml,create_directories
from src.transaction_fraud_detection.constants import *
from src.transaction_fraud_detection.entity.config_entity import DataIngestionConfig,DataValidationConfig,DataTransformationConfig


In [6]:
class ConfigManager:
    def __init__(
        self,
        config_file=CONFIG_FILE_PATH,
        schema_file=SCHEMA_FILE_PATH,
        params_file=PARAM_FILE_PATH):

        self.config=read_yaml(config_file)
        self.schema=read_yaml(schema_file)
        self.params=read_yaml(params_file)

        create_directories([self.config.artifacts_root])
    

    def get_data_tranformation_config(self)->DataTransformationConfig:
        config=self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config=DataIngestionConfig(
            root_dir=config.root_dir,
            data_path=config.data_path
        )
        return data_transformation_config

In [7]:
import os
import pandas as pd
from src.transaction_fraud_detection.logging import logger
from src.transaction_fraud_detection.config.configuration import ConfigManager
from src.transaction_fraud_detection.entity.config_entity import DataTransformationConfig
from sklearn.model_selection import train_test_split

[2025-08-16 18:46:20,791 : INFO : utils : NumExpr defaulting to 12 threads.]


In [8]:
class DataTranformation:
    def __init__(self,config:DataTransformationConfig):
        self.config=config
    
    def train_test_split(self):
        try:
            data=pd.read_csv(self.config.data_path)

            train,test=train_test_split(data,test_size=0.3,random_state=42)

            train.to_csv(os.path.join(self.config.root_dir,"train.csv"),index=False)
            test.to_csv(os.path.join(self.config.root_dir,"test.csv"),index=False)

            logger.info("data splitted sucessfully")
            logger.info(train.shape)
            logger.info(test.shape)

        except Exception as e:
            raise e

In [9]:
try:
    config=ConfigManager()
    data_tranformation_config=config.get_data_ingestion_config()
    data_tansformation=DataTranformation(config=data_tranformation_config)
    data_tansformation.train_test_split()
except Exception as e:
    raise e

[2025-08-16 18:48:39,114 : INFO : common : yaml_fileconfig\config.yaml loaded sucessfully]
[2025-08-16 18:48:39,116 : INFO : common : yaml_fileschema.yaml loaded sucessfully]
[2025-08-16 18:48:39,117 : INFO : common : yaml_fileparams.yaml loaded sucessfully]
[2025-08-16 18:48:39,118 : INFO : common : create directory at artifacts]
[2025-08-16 18:48:39,118 : INFO : common : create directory at artifacts/data_ingestion]


AttributeError: 'DataIngestionConfig' object has no attribute 'data_path'